In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import sys; sys.path.append('/content/drive/MyDrive/CS3268_Project/src')
import numpy as np, pandas as pd
from common import load_split, features, train_model, threshold_at_fpr, save_preds, BEST_PARAMS
np.random.seed(42)
print(BEST_PARAMS)  # expect n_estimators=232, max_depth=4, lr=0.1, mcw=10, subsample=0.8, colsample=0.8

In [ ]:
# Kamiran & Calders (2012) reweighing: w(g,y) = P(g) * P(y) / P(g,y), computed on TRAIN only.
# Makes group and label statistically independent in the weighted training set. No tuning, no sweep.
tr, va, te = load_split("train"), load_split("val"), load_split("test")
g, y = tr["age_group"], tr["fraud_bool"]
pg, py = g.value_counts(normalize=True), y.value_counts(normalize=True)
pgy = pd.crosstab(g, y, normalize=True)
W = pd.DataFrame({(a, b): pg[a] * py[b] / pgy.loc[a, b] for a in pg.index for b in py.index}, index=["w"]).T
print(W)  # expect: 50-plus fraud weight < 1 (high base rate), under-50 fraud weight > 1
w = np.array([W.loc[(a, b), "w"] for a, b in zip(g, y)])
print("mean weight (expect ~1.0):", w.mean())

In [ ]:
model = train_model(features(tr), y, sample_weight=w)   # frozen BEST_PARAMS, seed 42
s_va = model.predict_proba(features(va))[:, 1]
s_te = model.predict_proba(features(te))[:, 1]
cut = threshold_at_fpr(s_va, va["fraud_bool"], fpr=0.05)   # set on val only, applied unchanged to test
print("cutoff:", cut)
print(save_preds("reweigh", pd.concat([va, te], ignore_index=True), np.r_[s_va, s_te], cut))
model.save_model("/content/drive/MyDrive/CS3268_Project/models/reweigh.json")

In [ ]:
# Paste this output to Claude.
p = pd.read_parquet("/content/drive/MyDrive/CS3268_Project/preds/preds_reweigh.parquet")
rows = []
for name, d in [("val", p[p.month == 5]), ("test", p[p.month >= 6])]:
    for grp in (0, 1, "all"):
        s = d if grp == "all" else d[d.age_group == grp]
        h, f = s[s.y_true == 0], s[s.y_true == 1]
        rows.append(dict(split=name, group=grp, n=len(s), FPR=h.decision.mean(), recall=f.decision.mean()))
r = pd.DataFrame(rows).round(4); print(r)
for name in ("val", "test"):
    f0, f1 = r[(r.split == name) & (r.group == 0)].FPR.item(), r[(r.split == name) & (r.group == 1)].FPR.item()
    print(name, "FPR ratio (lower/higher):", round(min(f0, f1) / max(f0, f1), 3), "| flagged more:", "50-plus" if f1 > f0 else "under-50")